# 介紹

使用pyautogui來完成小恐龍的遊戲機器人。
小恐龍是在chrome中使用 "chrome://dino"啟動


In [ ]:
import pyautogui as pyg
from mss import mss
import time
import numpy as np

print("請在 3 秒內切換至小恐龍遊戲視窗...")
time.sleep(3)
pyg.press('space')  # 啟動遊戲

# ================= 參數設定區 =================
# 恐龍中心座標
DINO_X = 269
DINO_Y = 613

# 偵測距離設定 (控制反應時間的關鍵)
BASE_OFFSET = 90      # 【調快反應】初始距離加大，讓前期提早反應 (原本70)
MAX_OFFSET = 250      # 【調快反應】後期最大距離加大 (原本220)
# ==============================================

start_time = time.time()

with mss() as sct:
    while True:
        run_time = time.time() - start_time
        
        # 【調快反應】倍率從 1.5 改為 2.5，讓視線隨遊戲加速推得更快
        dynamic_offset = int(BASE_OFFSET + (run_time * 2.5))
        dynamic_offset = min(dynamic_offset, MAX_OFFSET) 
        
        monitor = {
            "left": DINO_X + dynamic_offset,
            "top": DINO_Y - 40,   # 上半部涵蓋半空中的鳥
            "width": 50, 
            "height": 55          # 涵蓋鳥與仙人掌，避開地板
        }

        sct_img = sct.grab(monitor)
        pixels = np.array(sct_img)[:, :, :3].astype(int)
        bg_color = pixels[0, 0]    
        diff = np.abs(pixels - bg_color)
        
        # 切分上下半部
        # 上半部 (高度 0~25)：專門看半空中的翼手龍
        bird_zone = diff[:25, :, :]
        # 下半部 (高度 25~55)：專門看地上的仙人掌與貼地飛行的鳥
        cactus_zone = diff[25:, :, :]
        
        # 優先判斷下半部 (地上的障礙物一定要跳)
        if np.any(cactus_zone > 30):
            pyg.keyDown('space')
            time.sleep(0.05) 
            pyg.keyUp('space')
            
            time.sleep(0.12)  
             
            pyg.keyDown('down')
            time.sleep(0.08)
            pyg.keyUp('down')
            
        # 若下方沒東西，上方有東西 -> 趴下躲避
        elif np.any(bird_zone > 30):
            pyg.keyDown('down')
            time.sleep(0.20)  # 趴下時間稍微縮短，讓他能更快起身應對下一個
            pyg.keyUp('down')      

請在 3 秒內切換至小恐龍遊戲視窗...


/var/folders/vd/vqm6vfxj1p7726w9q0yvdp2w0000gn/T/ipykernel_70497/882581792.py:22: DeprecationWarning: mss.mss is deprecated and will be removed in a future release; use mss.MSS instead
  with mss() as sct:
